# EXP — Enhancement Classical vs Restorasi Deep Learning

**Eksperimen pembanding TERISOLASI (sekali-jalan) — BUKAN metode utama.**
Skripsi FotoQu · Ade · TI UIN Alauddin Makassar

## Hipotesis
> Apakah restorasi DL *off-the-shelf* (CodeFormer) menghasilkan clustering lebih baik
> dibanding enhancement classical, dan berapa biaya compute-nya?

Tujuan: **memperkuat justifikasi pilihan classical** untuk sidang. Jika DL kalah →
*"restorer DL off-the-shelf tidak transfer ke objektif face-embedding tanpa fine-tuning,
dengan biaya compute jauh lebih tinggi"* — **bukan** "DL buruk". Jika DL menang → laporkan
selisih akurasi vs selisih compute eksplisit.

## Perbandingan terkontrol
Yang **berbeda** antar arm HANYA *enhancement*. **Identik**: face set, alignment,
extractor ArcFace, parameter HDBSCAN, subset.

| Arm | Enhancement |
|---|---|
| `control` | tidak ada (baseline) |
| `classical` | FQA-routed classical CV (Bicubic SR, NLM, Gamma, CLAHE, Laplacian) |
| `dl_w0.5/0.7/0.9` | CodeFormer, fidelity weight w |

### Alur data (align-ke-512 bersama)
```
foto subset ─detect(buffalo_l)→ bbox + kps           ← SHARED face set
   ├─ native img[bbox] ─→ FQA (resolusi NATIVE)       (hormati aturan CLAUDE.md)
   └─ norm_crop(img,kps,512) ─→ aligned512 (BGR)      ← SHARED alignment
         ├ control : identitas
         ├ classical: route_enhancement()   ─→ resize 512→112 → ArcFace get_feat → L2-norm
         └ dl(w)   : codeformer(aligned,w)                     └→ HDBSCAN (mcs,ms identik) → metrik
```

**Justifikasi (aturan #1 & #3 CLAUDE.md):**
- FQA di **crop native** `img[bbox]` → patuhi bug-terlarang "jangan resize sebelum skor blur".
- Align ke **512**: alignment identik antar arm; CodeFormer memang dirancang untuk wajah
  aligned 512 (input idealnya); detail native terjaga.
- **Bicubic SR** = `INTER_CUBIC` pada warp untuk wajah `res_s<0.7` (classical saja).
- HDBSCAN atas embedding **L2-normalized**, `metric='euclidean'` (≡ cosine).

**Keterbatasan (dicatat jujur):**
1. Classical di sini = **reimplementasi dari spec CLAUDE.md**, bukan berkas kode asli Ade.
2. Proxy label = clustering NB05 (tanpa enhancement) → indikatif, bukan ground-truth manusia.
3. Silhouette/DBI internal bias lintas-ruang → utamakan Purity/ARI/NMI eksternal.


## 0. Setup — dependencies & GPU
Deteksi+ArcFace: **GPU** (onnxruntime-gpu). CodeFormer: **GPU** (torch). FQA+classical:
**CPU** O(H·W)/wajah. Set runtime Colab ke **GPU (T4 cukup)**.

In [1]:
# ~3-5 menit. Runtime: GPU.
!pip install -q insightface onnxruntime-gpu opencv-python-headless pillow pillow-heif rawpy hdbscan scikit-learn tqdm
!pip install -q basicsr facexlib
!git clone -q https://github.com/sczhou/CodeFormer.git

# Fix: basicsr import torchvision.transforms.functional_tensor (dihapus di torchvision baru)
import sys, types, os, subprocess
try:
    import torchvision.transforms.functional_tensor  # noqa
except ModuleNotFoundError:
    import torchvision.transforms.functional as _F
    shim = types.ModuleType("torchvision.transforms.functional_tensor")
    shim.rgb_to_grayscale = _F.rgb_to_grayscale
    sys.modules["torchvision.transforms.functional_tensor"] = shim
    print("Applied torchvision.functional_tensor shim.")

subprocess.run("cd CodeFormer && python basicsr/setup.py develop 2>/dev/null", shell=True)
os.makedirs("CodeFormer/weights/CodeFormer", exist_ok=True)
if not os.path.exists("CodeFormer/weights/CodeFormer/codeformer.pth"):
    !wget -q https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer.pth -O CodeFormer/weights/CodeFormer/codeformer.pth
print("Bobot CodeFormer siap.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 762.2/762.2 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.3/220.3 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 54.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 kB 14.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 15.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.3/338.3 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
import onnxruntime as ort, torch
print("ONNX providers :", ort.get_available_providers())
print("CUDA (torch)   :", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Aktifkan GPU: Runtime > Change runtime type > GPU"
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ONNX providers : ['AzureExecutionProvider', 'CPUExecutionProvider']
CUDA (torch)   : True | Tesla T4
Mounted at /content/drive


## 1. Konfigurasi
Parameter WAJIB identik antar arm: `ALIGN_SIZE`, extractor buffalo_l, `MCS_SUBSET`/`MS_SUBSET`.

In [3]:
from pathlib import Path
import numpy as np

DATA_DIR = Path("/content/drive/MyDrive/OTW S.KOM/Genbi")
NB01_DIR = Path("/content/drive/MyDrive/OTW S.KOM/Embeddings/output_nb01")
NB05_DIR = Path("/content/drive/MyDrive/OTW S.KOM/Embeddings/output_nb05")
OUT_DIR  = Path("/content/drive/MyDrive/OTW S.KOM/Embeddings/output_exp_classical_vs_dl")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DET_SIZE, DET_THRESH = (640, 640), 0.35          # identik NB01
MCS_NB05, MS_NB05    = 50, 5                      # proxy identitas (reproduksi NB05)
N_IDENTITIES, MAX_PER_ID, RANDOM_SEED = 25, 40, 42
ALIGN_SIZE, ARCFACE_SIZE = 512, 112

# FQA thresholds (PLACEHOLDER — belum dikalibrasi, sesuai CLAUDE.md)
TH_RES, TH_NOISE, TH_ILLUM, TH_CONTRAST, TH_BLUR = 0.7, 0.6, 0.7, 0.6, 0.6
VARLAP_REF, NOISE_REF, ILLUM_SIGMA, CONTRAST_REF = 300.0, 15.0, 50.0, 60.0

CODEFORMER_W = [0.5, 0.7, 0.9]
MCS_SUBSET, MS_SUBSET = 10, 5                     # diskala utk subset kecil; identik semua arm

np.random.seed(RANDOM_SEED)
print("Subset target ≈", N_IDENTITIES, "identitas ×", MAX_PER_ID, "wajah")

Subset target ≈ 25 identitas × 40 wajah


## 2. Muat embedding NB01 + proxy label
Proxy identitas = HDBSCAN penuh (mcs=50,ms=5) atas embedding NB01 (tanpa enhancement).
Muat `output_nb05` bila ada; else reproduksi (deterministik).

In [4]:
import pickle, hdbscan
embeddings = np.load(NB01_DIR / "embeddings.npy").astype(np.float32)
with open(NB01_DIR / "metadata.pkl", "rb") as f:
    metadata = pickle.load(f)
assert len(embeddings) == len(metadata)
print(f"NB01: {embeddings.shape[0]} wajah × {embeddings.shape[1]} dim")

nb05_meta = NB05_DIR / "metadata_labeled.pkl"
if nb05_meta.exists():
    with open(nb05_meta, "rb") as f:
        proxy_labels = np.array([m["cluster_id"] for m in pickle.load(f)])
    print("Proxy label dari NB05.")
else:
    print("Reproduksi HDBSCAN(mcs=50, ms=5, euclidean)...")
    proxy_labels = hdbscan.HDBSCAN(min_cluster_size=MCS_NB05, min_samples=MS_NB05,
                                   metric="euclidean", core_dist_n_jobs=-1).fit_predict(embeddings)
n_clu = len(set(proxy_labels)) - (1 if -1 in proxy_labels else 0)
print(f"Proxy identitas: {n_clu} cluster, noise={(proxy_labels==-1).sum()}")

NB01: 11902 wajah × 512 dim
Proxy label dari NB05.
Proxy identitas: 144 cluster, noise=909


## 3. Subset stratified
Pilih `N_IDENTITIES` cluster stratified-by-size (campur besar–kecil), cap `MAX_PER_ID`/id.

In [5]:
from collections import Counter
sizes = Counter(proxy_labels[proxy_labels >= 0])
ordered = [lbl for lbl, _ in sizes.most_common()]
chosen = ordered if len(ordered) <= N_IDENTITIES else \
    [ordered[i] for i in np.linspace(0, len(ordered)-1, N_IDENTITIES).round().astype(int)]

rng = np.random.default_rng(RANDOM_SEED)
subset = []
for lbl in chosen:
    rows = np.where(proxy_labels == lbl)[0]
    if len(rows) > MAX_PER_ID:
        rows = rng.choice(rows, MAX_PER_ID, replace=False)
    for r in rows:
        m = metadata[r]
        subset.append({"photo_path": m["photo_path"], "bbox": m["bbox"], "proxy_id": int(lbl)})
print(f"Subset: {len(subset)} wajah, {len(chosen)} identitas, "
      f"{len(set(s['photo_path'] for s in subset))} foto unik")

Subset: 1000 wajah, 25 identitas, 438 foto unik


## 4. Model & helper (deteksi, IoU, baca gambar, warp 512)

In [6]:
import cv2
from insightface.app import FaceAnalysis
from insightface.utils import face_align
from pillow_heif import register_heif_opener; register_heif_opener()

app = FaceAnalysis(name="buffalo_l", allowed_modules=["detection", "recognition"])
app.prepare(ctx_id=0, det_size=DET_SIZE, det_thresh=DET_THRESH)
rec_model = app.models["recognition"]        # ArcFaceONNX — identik semua arm
print("buffalo_l siap.")

def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/ua if ua > 0 else 0.0

def imread_any(p):
    img = cv2.imread(p)
    if img is not None: return img
    ext = Path(p).suffix.lower()
    if ext in {'.raf','.cr2','.nef','.arw','.dng','.rw2','.orf'}:
        import rawpy
        with rawpy.imread(p) as raw:
            return cv2.cvtColor(raw.postprocess(), cv2.COLOR_RGB2BGR)
    from PIL import Image
    return cv2.cvtColor(np.array(Image.open(p).convert("RGB")), cv2.COLOR_RGB2BGR)

def warp512(img, kps, cubic=False):
    M = face_align.estimate_norm(kps, image_size=ALIGN_SIZE)   # 512 % 128 == 0 → valid
    flag = cv2.INTER_CUBIC if cubic else cv2.INTER_LINEAR
    return cv2.warpAffine(img, M, (ALIGN_SIZE, ALIGN_SIZE), flags=flag, borderValue=0.0)

download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:05<00:00, 48466.54KB/s]
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:147: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_l/genderage.onnx genderage
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
buffalo_l siap.


## 5. FQA — dari spec CLAUDE.md (dihitung di crop NATIVE)

| Komponen | Metode | Referensi |
|---|---|---|
| Blur | Variance of Laplacian | Pech-Pacheco et al., 2000 |
| Resolusi | min(bbox)/112 | ad-hoc |
| Illumination | Gaussian(mean)+minmax(contrast) | Zuiderveld, 1994 |
| Noise | Immerkær σ | Immerkær, 1996 |
| Pose | yaw ratio 5-titik | ad-hoc (validasi terbatas) |

Sub-skor granular disimpan terpisah (tak di-collapse untuk routing). O(H·W), CPU.

In [7]:
import math
def immerkaer_sigma(gray):
    M = np.array([[1,-2,1],[-2,4,-2],[1,-2,1]], dtype=np.float64)
    H, W = gray.shape
    conv = cv2.filter2D(gray.astype(np.float64), -1, M)
    return math.sqrt(math.pi/2) * np.abs(conv).sum() / (6.0*max(1,W-2)*max(1,H-2))

def tenengrad(gray):     # metrik sharpening INDEPENDEN (Sobel) — bukan var-Laplacian
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    return float((gx**2 + gy**2).mean())

def compute_sub_scores(native_bgr, bbox, kps):
    gray = cv2.cvtColor(native_bgr, cv2.COLOR_BGR2GRAY)
    var_lap = cv2.Laplacian(gray, cv2.CV_64F).var()
    blur_s = float(np.clip(var_lap/VARLAP_REF, 0, 1))
    bw, bh = bbox[2]-bbox[0], bbox[3]-bbox[1]
    res_s = float(np.clip(min(bw, bh)/112.0, 0, 1))
    mean_b = float(gray.mean())
    illum_mean_s = float(math.exp(-((mean_b-128.0)**2)/(2*ILLUM_SIGMA**2)))
    contrast_s = float(np.clip(gray.std()/CONTRAST_REF, 0, 1))
    sigma = immerkaer_sigma(gray)
    noise_s = float(np.clip(1.0 - sigma/NOISE_REF, 0, 1))
    le, re, nose = kps[0], kps[1], kps[2]
    d_l, d_r = abs(nose[0]-le[0]), abs(re[0]-nose[0])
    yaw = min(d_l, d_r)/max(d_l, d_r) if max(d_l, d_r) > 0 else 0.0
    return {"blur": blur_s, "resolution": res_s,
            "illumination": 0.5*illum_mean_s + 0.5*contrast_s, "noise": noise_s,
            "pose": float(yaw), "_mean_brightness_raw": mean_b,
            "_contrast_score": contrast_s, "_illum_mean_s": illum_mean_s}

## 6. Deteksi, alignment, & FQA (satu pass, SHARED)
Deteksi (identik NB01) → cocokkan bbox subset via IoU → `kps` → `norm_crop` 512 (linear,
SHARED). FQA di native. Precompute warp **cubic** hanya untuk wajah `res_s<0.7` (basis SR
classical) — hemat RAM.

In [8]:
from tqdm.notebook import tqdm
from collections import defaultdict

by_photo = defaultdict(list)
for s in subset:
    by_photo[s["photo_path"]].append(s)

faces, miss = [], 0
for path, entries in tqdm(by_photo.items(), desc="Deteksi+align+FQA"):
    img = imread_any(path)
    if img is None:
        miss += len(entries); continue
    H, W = img.shape[:2]
    dets = app.get(img)
    for e in entries:
        best, best_iou = None, 0.0
        for d in dets:
            i = iou(e["bbox"], d.bbox.tolist())
            if i > best_iou: best, best_iou = d, i
        if best is None or best_iou < 0.5:
            miss += 1; continue
        x1, y1, x2, y2 = [int(v) for v in best.bbox]
        x1, y1, x2, y2 = max(0,x1), max(0,y1), min(W,x2), min(H,y2)
        native = img[y1:y2, x1:x2]
        if native.size == 0:
            miss += 1; continue
        fqa = compute_sub_scores(native, [x1,y1,x2,y2], best.kps)
        rec = {"proxy_id": e["proxy_id"], "fqa": fqa,
               "aligned512": warp512(img, best.kps, cubic=False)}
        if fqa["resolution"] < TH_RES:                     # basis SR (cubic) hanya bila perlu
            rec["aligned512_cubic"] = warp512(img, best.kps, cubic=True)
        faces.append(rec)
proxy_ids = np.array([f["proxy_id"] for f in faces])
print(f"Wajah siap: {len(faces)} | gagal cocok: {miss} | "
      f"perlu SR: {sum('aligned512_cubic' in f for f in faces)}")

Deteksi+align+FQA:   0%|          | 0/438 [00:00<?, ?it/s]

Wajah siap: 1000 | gagal cocok: 0 | perlu SR: 259


## 7. Enhancement classical — urutan WAJIB SR→NLM→Illum→Sharpen
Semua di kanvas 512, CPU. Trigger = sub-skor (threshold placeholder). Illumination
(gamma & CLAHE) dievaluasi pakai skor terpisah `_illum_mean_s` & `_contrast_score`
(hindari dual-gating). SR = basis cubic yang sudah di-precompute.

In [9]:
def apply_nlm_denoising(img):
    return cv2.fastNlMeansDenoisingColored(img, None, 5, 5, 7, 21)

def apply_gamma_correction(img, mean_raw):
    m = float(np.clip(mean_raw/255.0, 1e-3, 0.999))
    gamma = float(np.clip(math.log(0.5)/math.log(m), 0.4, 2.5))
    lut = np.array([((i/255.0)**(1.0/gamma))*255 for i in range(256)]).astype("uint8")
    return cv2.LUT(img, lut)

def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)

def apply_laplacian_sharpening(img):
    k = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32)
    return cv2.filter2D(img, -1, k)

def route_enhancement(f):
    s = f["fqa"]
    img = f.get("aligned512_cubic", f["aligned512"]).copy()   # SR: cubic bila res_s<0.7
    if s["noise"] < TH_NOISE:            img = apply_nlm_denoising(img)
    if s["_illum_mean_s"] < TH_ILLUM:    img = apply_gamma_correction(img, s["_mean_brightness_raw"])
    if s["_contrast_score"] < TH_CONTRAST: img = apply_clahe(img)
    if s["blur"] < TH_BLUR:              img = apply_laplacian_sharpening(img)
    return img

## 8. CodeFormer — loader & restorasi
**Bypass** FaceRestoreHelper (detektor internalnya) → jalankan hanya jaring restorasi pada
`aligned512` kita agar face set + alignment identik. `w` tinggi = setia; rendah = lebih
menghalusinasi (bisa menggeser identitas). GPU.

In [13]:
import sys, types, torch

# 1. Flush cached basicsr
for _k in list(sys.modules):
    if _k == 'basicsr' or _k.startswith('basicsr.'):
        del sys.modules[_k]

# 2. Namespace stub — skip basicsr/__init__.py (loads .losses → lpips yang tidak ada)
_basicsr_ns = types.ModuleType("basicsr")
_basicsr_ns.__path__ = ["CodeFormer/basicsr"]
_basicsr_ns.__package__ = "basicsr"
sys.modules["basicsr"] = _basicsr_ns

# 3. CodeFormer di depan sys.path
sys.path = ["CodeFormer"] + [p for p in sys.path if p != "CodeFormer"]

from basicsr.archs.codeformer_arch import CodeFormer as CodeFormerNet
from basicsr.utils import img2tensor, tensor2img
from torchvision.transforms.functional import normalize

device = "cuda"
cf_net = CodeFormerNet(dim_embd=512, codebook_size=1024, n_head=8,
    n_layers=9, connect_list=["32","64","128","256"]).to(device)
cf_net.load_state_dict(torch.load("CodeFormer/weights/CodeFormer/codeformer.pth")["params_ema"])
cf_net.eval(); print("CodeFormer dimuat.")

@torch.no_grad()
def codeformer_restore(img512_bgr, w):
    t = img2tensor(img512_bgr/255.0, bgr2rgb=True, float32=True)
    normalize(t, (0.5,0.5,0.5), (0.5,0.5,0.5), inplace=True)
    out = cf_net(t.unsqueeze(0).to(device), w=w, adain=True)[0]
    return tensor2img(out, rgb2bgr=True, min_max=(-1,1)).astype("uint8")

CodeFormer dimuat.


## 9. Jalankan semua arm → embedding + biaya compute
Embedding: `rec_model.get_feat(crop112)` → L2-norm (identik). Waktu diukur **hanya di
langkah enhancement**. Peak GPU (torch) hanya menangkap CodeFormer, bukan ArcFace
(onnxruntime, alokator terpisah) → pemisahan biaya bersih.

In [14]:
import time
def embed(crop512_bgr):
    c = cv2.resize(crop512_bgr, (ARCFACE_SIZE, ARCFACE_SIZE), interpolation=cv2.INTER_AREA)
    e = rec_model.get_feat(c).flatten().astype(np.float32)
    return e / (np.linalg.norm(e) + 1e-9)

results = {}
def run_arm(name, enhance_fn, use_gpu):
    if use_gpu:
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    embs, t_enh = [], 0.0
    for f in tqdm(faces, desc=name):
        t0 = time.perf_counter()
        crop = enhance_fn(f)
        t_enh += time.perf_counter() - t0
        embs.append(embed(crop))
    peak = torch.cuda.max_memory_allocated()/1e6 if use_gpu else 0.0
    results[name] = {"embeddings": np.vstack(embs), "enh_time_total": t_enh,
                     "enh_time_mean_ms": 1000*t_enh/len(faces), "peak_mem_mb": peak}
    print(f"  {name}: total={t_enh:.1f}s, {1000*t_enh/len(faces):.1f} ms/wajah, peak GPU={peak:.0f} MB")

run_arm("control",   lambda f: f["aligned512"],        use_gpu=False)
run_arm("classical", route_enhancement,                use_gpu=False)
for w in CODEFORMER_W:
    run_arm(f"dl_w{w}", (lambda ww: lambda f: codeformer_restore(f["aligned512"], ww))(w), use_gpu=True)

control:   0%|          | 0/1000 [00:00<?, ?it/s]

  control: total=0.0s, 0.0 ms/wajah, peak GPU=0 MB


classical:   0%|          | 0/1000 [00:00<?, ?it/s]

  classical: total=2.8s, 2.8 ms/wajah, peak GPU=0 MB


dl_w0.5:   0%|          | 0/1000 [00:00<?, ?it/s]

  dl_w0.5: total=228.9s, 228.9 ms/wajah, peak GPU=849 MB


dl_w0.7:   0%|          | 0/1000 [00:00<?, ?it/s]

  dl_w0.7: total=220.8s, 220.8 ms/wajah, peak GPU=849 MB


dl_w0.9:   0%|          | 0/1000 [00:00<?, ?it/s]

  dl_w0.9: total=215.3s, 215.3 ms/wajah, peak GPU=849 MB


## 10. Clustering (HDBSCAN identik) + metrik
Internal: Silhouette, DBI. Eksternal vs proxy: Purity, ARI, NMI. + noise/coverage/compute.

In [15]:
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                                  adjusted_rand_score, normalized_mutual_info_score)
def purity(y_true, y_pred):
    mask = y_pred >= 0
    if mask.sum() == 0: return 0.0
    yt, yp = y_true[mask], y_pred[mask]
    return sum(np.bincount(yt[yp==c]).max() for c in set(yp)) / mask.sum()

def evaluate(name, X):
    lab = hdbscan.HDBSCAN(min_cluster_size=MCS_SUBSET, min_samples=MS_SUBSET,
                          metric="euclidean", core_dist_n_jobs=-1).fit_predict(X)
    m = X.shape[0]; nc = len(set(lab)) - (1 if -1 in lab else 0)
    nn = int((lab == -1).sum()); cl = lab >= 0
    ok = nc > 1 and cl.sum() > nc
    sil = silhouette_score(X[cl], lab[cl]) if ok else np.nan
    dbi = davies_bouldin_score(X[cl], lab[cl]) if ok else np.nan
    r = results[name]
    return {"arm": name, "n_clusters": nc, "coverage": round((m-nn)/m,3),
            "noise_rate": round(nn/m,3),
            "silhouette": round(float(sil),4) if sil==sil else None,
            "dbi": round(float(dbi),4) if dbi==dbi else None,
            "purity": round(purity(proxy_ids, lab),4),
            "ARI": round(adjusted_rand_score(proxy_ids, lab),4),
            "NMI": round(normalized_mutual_info_score(proxy_ids, lab),4),
            "enh_ms/face": round(r["enh_time_mean_ms"],2),
            "enh_total_s": round(r["enh_time_total"],1),
            "peak_GPU_MB": round(r["peak_mem_mb"],0)}

rows = [evaluate(n, results[n]["embeddings"]) for n in results]

In [16]:
import pandas as pd
df = pd.DataFrame(rows).set_index("arm")[
    ["n_clusters","coverage","noise_rate","silhouette","dbi",
     "purity","ARI","NMI","enh_ms/face","enh_total_s","peak_GPU_MB"]]
df.to_csv(OUT_DIR / "hasil_perbandingan.csv")
print("Tersimpan:", OUT_DIR / "hasil_perbandingan.csv"); df

Tersimpan: /content/drive/MyDrive/OTW S.KOM/Embeddings/output_exp_classical_vs_dl/hasil_perbandingan.csv


,n_clusters,coverage,noise_rate,silhouette,dbi,purity,ARI,NMI,enh_ms/face,enh_total_s,peak_GPU_MB
arm,,,,,,,,,,,
control,28,0.964,0.036,0.3569,1.3348,0.9855,0.9061,0.9526,0.00,0.0,0.0
classical,27,0.936,0.064,0.3557,1.2958,0.9893,0.8830,0.9444,2.78,2.8,0.0
dl_w0.5,28,0.915,0.085,0.2972,1.4958,0.9978,0.8329,0.9277,228.92,228.9,849.0
dl_w0.7,28,0.929,0.071,0.3185,1.4460,0.9978,0.8655,0.9393,220.80,220.8,849.0
dl_w0.9,29,0.938,0.062,0.3328,1.4095,0.9968,0.8699,0.9385,215.32,215.3,849.0


## 11. Interpretasi (isi setelah run)
- **Purity/ARI/NMI** lebih adil lintas-preprocessing daripada Silhouette/DBI internal.
- Classical ≳ DL pada Purity dengan `enh_ms/face` & `peak_GPU_MB` jauh lebih kecil →
  **justifikasi kuat memilih classical**.
- DL > classical → laporkan **selisih akurasi vs selisih compute** eksplisit.
- Efek **w**: w rendah sering **menurunkan** Purity (halusinasi menggeser identitas) — catat.
- **Wajib di skripsi:** eksperimen terkontrol sekali-jalan, subset proxy-labeled; bukan
  pengganti Skenario A/B/C; threshold FQA masih placeholder.

In [17]:
bp = df["purity"].idxmax(); ch = df["enh_ms/face"].idxmin()
print(f"Purity tertinggi   : {bp} ({df.loc[bp,'purity']})")
print(f"Enhancement tercepat: {ch} ({df.loc[ch,'enh_ms/face']} ms/wajah)")
if "classical" in df.index and "dl_w0.7" in df.index:
    print(f"classical vs dl_w0.7 → purity {df.loc['classical','purity']} vs {df.loc['dl_w0.7','purity']}"
          f" | {df.loc['classical','enh_ms/face']} vs {df.loc['dl_w0.7','enh_ms/face']} ms/wajah"
          f" | GPU {df.loc['classical','peak_GPU_MB']} vs {df.loc['dl_w0.7','peak_GPU_MB']} MB")

Purity tertinggi   : dl_w0.5 (0.9978)
Enhancement tercepat: control (0.0 ms/wajah)
classical vs dl_w0.7 → purity 0.9893 vs 0.9978 | 2.78 vs 220.8 ms/wajah | GPU 0.0 vs 849.0 MB
